In [3]:
import pandas as pd
from datasets import Dataset
from datasets import ClassLabel
from google.colab import drive
drive.mount('/drive')


train_df = pd.read_csv("/drive/MyDrive/train_banking77.csv")
test_df = pd.read_csv("/drive/MyDrive/test_Banking77.csv")



train_dataset = Dataset.from_pandas(train_df)



num_labels = train_df['label'].nunique()

train_dataset = train_dataset.cast_column(
    "label", ClassLabel(num_classes=num_labels)
)

dataset_split = train_dataset.train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column="label"
)

dataset_split["validation"] = dataset_split["test"]
del dataset_split["test"]

Mounted at /drive


Casting the dataset:   0%|          | 0/10003 [00:00<?, ? examples/s]

In [ ]:
## Model Selection

'''
Task:
Multi-class Intent Classification

Candidate approaches:
1. TF-IDF + Logistic Regression
2. DistilBERT
3. RoBERTa-base

Final transformer model:
RoBERTa-base
'''

In [1]:
from transformers import AutoTokenizer

checkpoint = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [2]:
text = "I lost my bank card"

encoded = tokenizer(text)

encoded

{'input_ids': [0, 100, 685, 127, 827, 1886, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [4]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True
    )

In [5]:
tokenized_dataset = dataset_split.map(
    tokenize_function
)

tokenized_dataset

Map:   0%|          | 0/9002 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text', 'input_ids', 'attention_mask'],
        num_rows: 9002
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text', 'input_ids', 'attention_mask'],
        num_rows: 1001
    })
})

In [6]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [7]:
from transformers import AutoModelForSequenceClassification

checkpoint = "roberta-base"

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=77
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
model

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [16]:
dataset_split.save_to_disk(
    "/drive/MyDrive/dataset_split"
)

Saving the dataset (0/1 shards):   0%|          | 0/9002 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1001 [00:00<?, ? examples/s]

In [17]:
tokenized_dataset.save_to_disk(
    "/drive/MyDrive/tokenized_dataset"
)

Saving the dataset (0/1 shards):   0%|          | 0/9002 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1001 [00:00<?, ? examples/s]